# Task 2 Evaluating Reweighing / 任务 2 评估重新加权

We implement **Reweighing from Calders, Kamiran and Pechenizkiy (2009)** and test it on three datasets. Following the paper, weights are used to sample training records with replacement. We report the required fairness scores for the original labels before training. We then compare an ordinary classifier and a classifier trained after Reweighing on the same held-out test set.

我们实现 Calders、Kamiran 和 Pechenizkiy（2009）的 Reweighing，并在三个数据集上评估。按照论文，权重用于对训练记录进行有放回抽样。先在训练模型前计算原始标签的公平性指标，再在同一份保留的测试集上比较普通分类器与重新加权抽样后训练的分类器。

**How to run / 如何运行**

Extract the ZIP and open this notebook in Jupyter or VS Code. Keep the `data` folder beside it and choose **Run All**. The files in `data` are already downloaded. If a file is missing, the notebook downloads it from its source. The only files the notebook writes are missing input datasets. Results are shown below as tables.

解压 ZIP，在 Jupyter 或 VS Code 中打开本文件，让 `data` 文件夹与它位于同一目录，然后选择 **Run All**。输入数据已经下载好。若缺少文件，代码会从来源下载。运行代码只会写入缺少的输入数据文件，实验结果直接显示为表格。

If imports fail, run the following once in a new code cell, then restart the kernel. / 若导入失败，在新代码单元格运行下面命令一次，然后重启内核。

```python
%pip install numpy pandas scipy "scikit-learn>=1.2" ipython
```

Tested package versions / 本次执行的包版本 — numpy 2.3.5, pandas 2.2.3, scipy 1.17.0, scikit-learn 1.8.0, ipython 9.16.1.

In [1]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
import hashlib

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score
from IPython.display import display

SEED = 42
TEST_SIZE = 0.30
N_NEIGHBORS = 5
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Fixed source checksums keep the input files reproducible.
SOURCES = {
    "adult.zip": (
        "https://archive.ics.uci.edu/static/public/2/adult.zip",
        "7537312dd56c2b98035880805ce99e68183a30ee468aa5329d6df0fbb3cc21bb",
    ),
    "german.zip": (
        "https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip",
        "e12d9d5def6845c0622634a1cd2ab87fa470668c4298f1ec52a4e403376a435b",
    ),
    "compas-scores-two-years.csv": (
        "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv",
        "c451db85908b2f7fef1d83203bedf6b71ecda0d5af468d82ae62178f91d0cc7d",
    ),
}


def input_file(name):
    path = DATA_DIR / name
    url, expected_hash = SOURCES[name]
    if path.exists():
        content = path.read_bytes()
    else:
        with urlopen(url, timeout=120) as response:
            content = response.read()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise ValueError(f"Source file differs from the version used here: {name}")
    if not path.exists():
        path.write_bytes(content)
    return content

## 1 Data and group definitions / 数据与分组

In every dataset, `Y=1` means a favourable recorded outcome. `A=0` is the comparison group listed as unprivileged and `A=1` is the reference group listed as privileged. These are the group definitions for this experiment.

三个数据集均将 `Y=1` 定义为有利的原始标签结果。`A=0` 对应下表的非优势比较组，`A=1` 对应优势参照组。这些是本次实验采用的分组定义。

| Dataset / 数据集 | A=0 | A=1 | Y=1 |
|---|---|---|---|
| Adult | Female / 女性 | Male / 男性 | Income >50K / 年收入超过五万美元 |
| German Credit | Female / 女性 | Male / 男性 | Good credit risk / 良好信用风险标签 |
| COMPAS | African-American | Caucasian | No recorded recidivism within two years / 标签显示两年内未再犯 |

- **Adult.** Combine `adult.data` and `adult.test`, since this task asks for a new stratified split. Keep all rows and treat `?` as a separate `Missing` category. Drop the census sampling weight `fnlwgt` and the text `education` field, which duplicates `education_num`. Exclude `sex` from model inputs. All other fields are retained.
- **German Credit.** Use the original symbolic file. Get sex from `personal_status` using UCI's codebook (`A91/A93/A94` male, `A92/A95` female). Remove that combined field from model inputs because it directly contains sex. Keep all other attributes and all rows.
- **COMPAS.** Apply ProPublica's filters: screening within ±30 days of arrest, `is_recid != -1`, charge degree other than `O`, and a valid score text. Keep the two race groups above. Use `age`, `sex`, the three juvenile offence counts, `priors_count`, and `c_charge_degree` as predictors. Exclude risk scores and later outcome fields to avoid using information that reveals or replaces the target. Race is used for fairness scores and weights.

- **Adult。** 合并原来的训练文件与测试文件，再按本任务要求重新分层划分。保留所有记录，把 `?` 作为单独的 `Missing` 类别。删除人口抽样权重 `fnlwgt` 和与教育年数重复的文字字段 `education`。模型不使用 `sex`，其余字段保留。
- **German Credit。** 使用原始分类编码文件。按 UCI 字典从 `personal_status` 提取性别，随后从模型输入中移除这个直接包含性别的组合字段。保留其余属性和所有记录。
- **COMPAS。** 使用 ProPublica 的上述筛选条件，并只保留所比较的两个种族组。模型使用年龄、性别、三项未成年犯罪计数、既往犯罪次数和指控等级。风险评分与后续结果字段不作为输入，以避免使用揭示或替代目标的信息。种族用于计算公平性指标和权重。

The original-label scores below refer to the retained population after these stated data choices. For German and COMPAS, the positive class describes a recorded label rather than a directly observed approval or release decision.

下文原始标签指标针对按上述规则保留的数据。German 和 COMPAS 的正类表示数据记录的标签，而不是直接观测到的贷款批准或释放决定。

Retained rows / 保留记录数 — **Adult 48,842; German Credit 1,000; COMPAS 5,278**. Training/test sizes / 训练与测试数量 — **34,189/14,653; 700/300; 3,694/1,584**, respectively.

In [2]:
def load_adult():
    columns = [
        "age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race", "sex",
        "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
    ]
    with ZipFile(BytesIO(input_file("adult.zip"))) as archive:
        parts = []
        for name in ["adult.data", "adult.test"]:
            part = pd.read_csv(
                archive.open(name), names=columns, skipinitialspace=True,
                comment="|", na_values="?",
            )
            parts.append(part)
    frame = pd.concat(parts, ignore_index=True)
    income = frame["income"].str.strip().str.rstrip(".")
    y = income.eq(">50K").astype(int).to_numpy()
    a = frame["sex"].eq("Male").astype(int).to_numpy()
    x = frame.drop(columns=["income", "sex", "fnlwgt", "education"])
    categorical = x.select_dtypes(exclude="number").columns
    x[categorical] = x[categorical].fillna("Missing")
    assert income.isin([">50K", "<=50K"]).all()
    assert frame["sex"].isin(["Female", "Male"]).all()
    return x, y, a


def load_german():
    columns = [
        "checking", "duration", "credit_history", "purpose", "credit_amount",
        "savings", "employment", "installment_rate", "personal_status",
        "other_debtors", "residence_since", "property", "age", "other_plans",
        "housing", "existing_credits", "job", "people_liable", "telephone",
        "foreign_worker", "credit_label",
    ]
    with ZipFile(BytesIO(input_file("german.zip"))) as archive:
        frame = pd.read_csv(archive.open("german.data"), sep=r"\s+", names=columns)
    sex_codes = {"A91": 1, "A93": 1, "A94": 1, "A92": 0, "A95": 0}
    a = frame["personal_status"].map(sex_codes)
    assert a.notna().all() and frame["credit_label"].isin([1, 2]).all()
    y = frame["credit_label"].eq(1).astype(int).to_numpy()
    x = frame.drop(columns=["credit_label", "personal_status"])
    return x, y, a.to_numpy(dtype=int)


def load_compas():
    frame = pd.read_csv(BytesIO(input_file("compas-scores-two-years.csv")))
    keep = (
        frame["days_b_screening_arrest"].between(-30, 30)
        & frame["is_recid"].ne(-1)
        & frame["c_charge_degree"].ne("O")
        & frame["score_text"].notna()
        & frame["score_text"].ne("N/A")
        & frame["race"].isin(["African-American", "Caucasian"])
    )
    frame = frame.loc[keep].reset_index(drop=True)
    predictors = [
        "age", "sex", "juv_fel_count", "juv_misd_count", "juv_other_count",
        "priors_count", "c_charge_degree",
    ]
    x = frame[predictors].copy()
    assert not x.isna().any().any()
    assert frame["two_year_recid"].isin([0, 1]).all()
    y = 1 - frame["two_year_recid"].to_numpy(dtype=int)
    a = frame["race"].eq("Caucasian").to_numpy(dtype=int)
    return x, y, a


datasets = {
    "Adult": load_adult(),
    "German Credit": load_german(),
    "COMPAS": load_compas(),
}

## 2 Original labels before training / 训练前的原始标签

Let $r_0$ and $r_1$ be the positive rates for `A=0` and `A=1`.

- **DP_Diff = $r_0-r_1$**. The ideal value is 0. A negative score means a lower positive rate for `A=0`. A smaller absolute value means a smaller group gap.
- **DIR = $r_0/r_1$**. The ideal value is 1. We compare distance from 1 rather than treating every larger value as better. If $r_1=0$, the code returns `NaN` for 0/0 or infinity for a positive numerator.
- **Consistency** uses the AIF360 formula $1-\frac{1}{n}\sum_i |y_i-\frac{1}{5}\sum_{j\in N_5(i)}y_j|$. Higher means that nearby records have more similar labels. We use Euclidean distance after standardising numeric columns and one-hot encoding categorical columns. As in AIF360, each record is queried against the same feature matrix, so self-matches are allowed. We use exact brute-force search. Sensitive attributes are excluded from this feature space. Ties follow scikit-learn's neighbour order.

令 $r_0$ 和 $r_1$ 分别表示两个群体的正类比例。DP_Diff 使用“非优势组减优势组”的有符号差，越接近 0 越好。DIR 为两组比例之比，越接近 1 越好。Consistency 按 AIF360 的公式计算，用标准化数值列和独热编码分类列之间的欧氏距离找五个邻居，数值越大表示相近记录的标签越一致。与 AIF360 一样，每条记录在同一特征矩阵中查询邻居，查询记录自身也允许成为邻居。这里使用精确的暴力搜索。距离特征不包含受保护属性，距离相同时按 scikit-learn 返回的顺序处理。

For this descriptive check, the encoder and scaler use the full retained feature data. This produces a metric for the original labels before a classifier is trained. A fresh encoder and scaler will be fitted using only training records for the model comparison.

这一步是训练前的描述性检查，使用全部保留记录的特征计算原始标签的一致性。后面的模型比较会重新创建编码器和标准化器，只在训练记录上拟合。

In [3]:
def make_preprocessor(x):
    numeric = x.select_dtypes(include="number").columns.tolist()
    categorical = x.select_dtypes(exclude="number").columns.tolist()
    return ColumnTransformer([
        ("numeric", StandardScaler(), numeric),
        ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])


def neighbours(features):
    search = NearestNeighbors(n_neighbors=N_NEIGHBORS, algorithm="brute", n_jobs=2)
    search.fit(features)
    return search.kneighbors(features, return_distance=False)


def fairness_scores(labels, groups, neighbour_indices):
    labels = np.asarray(labels, dtype=int)
    groups = np.asarray(groups, dtype=int)
    assert set(np.unique(labels)).issubset({0, 1})
    assert set(np.unique(groups)) == {0, 1}
    rate_0 = labels[groups == 0].mean()
    rate_1 = labels[groups == 1].mean()
    if rate_1 > 0:
        ratio = rate_0 / rate_1
    else:
        ratio = np.nan if rate_0 == 0 else np.inf
    consistency = 1 - np.abs(labels - labels[neighbour_indices].mean(axis=1)).mean()
    return {"DP_Diff": rate_0 - rate_1, "DIR": ratio, "Consistency": consistency}


original_rows = []
for name, (x, y, a) in datasets.items():
    metric_features = make_preprocessor(x).fit_transform(x)
    original_neighbours = neighbours(metric_features)
    original_rows.append({
        "Dataset": name,
        **fairness_scores(y, a, original_neighbours),
    })
original_results = pd.DataFrame(original_rows)
display(original_results.round(4))

,Dataset,DP_Diff,DIR,Consistency
0,Adult,-0.1945,0.3597,0.8432
1,German Credit,-0.0748,0.8966,0.7428
2,COMPAS,-0.1323,0.7828,0.6529


## 3 Stratified split and Reweighing / 分层划分与重新加权

We use **70% training data and 30% test data, seed 42**, stratifying by the four `A × Y` combinations. This keeps both groups and both labels represented in each split. The sensitive attribute is available to calculate training weights and fairness scores. It is excluded from the classifier inputs. Other features can still carry group information.

采用 **70% 训练、30% 测试、随机种子 42**，按 `A × Y` 四种组合分层，让两组和两种标签在划分中得到保留。受保护属性用于计算训练权重和公平性指标，不直接作为模型输入。其他特征仍可能包含与群体相关的信息。

For a training record in group $a$ with label $y$, use

$$w(a,y)=\frac{n_a n_y}{n\,n_{a,y}}.$$

All counts come from the training set. The code checks that the weighted training distribution has equal positive rates across groups. It then draws **n records with replacement** using probabilities proportional to these weights. The actual sampled counts may vary. The baseline uses all original training records once.

公式中的计数全部来自训练集。代码检查按精确权重计算的训练分布是否达到群体正类比例相等，随后按权重进行 **n 次有放回抽样**。实际抽样人数会有随机变化。普通模型使用原始训练记录各一次。

Both models use the same encoder and scaler fitted on the original training features, and the same **LogisticRegression(C=1.0, solver="liblinear", max_iter=2000, random_state=42)**. A probability of at least **0.5** gives a positive prediction. Parameters and the threshold are fixed before test evaluation. The test records and their original labels are used for evaluation. Consistency uses one shared neighbour graph built from the transformed test features, so both prediction sets are scored on the same records and distances.

两个模型共用在原始训练特征上拟合的编码器与标准化器，并采用上述相同的逻辑回归参数，正类概率达到 **0.5** 即预测为正类。参数与阈值在评估测试集之前固定。测试记录及其原始标签用于最终评估。两套预测的一致性共用同一份测试集邻居索引，因此比较中的记录与距离相同。

In [4]:
def reweighing_weights(y, a):
    n = len(y)
    weights = np.empty(n, dtype=float)
    for group in [0, 1]:
        for label in [0, 1]:
            group_mask = a == group
            label_mask = y == label
            joint_mask = group_mask & label_mask
            joint_count = joint_mask.sum()
            if joint_count == 0:
                raise ValueError("A group-label combination has no training records.")
            weights[joint_mask] = group_mask.sum() * label_mask.sum() / (n * joint_count)
    # Check the exact weighted distribution before random sampling.
    rates = [np.average(y[a == g], weights=weights[a == g]) for g in [0, 1]]
    assert np.isclose(rates[0], rates[1])
    assert np.isclose(weights.sum(), n)
    return weights


test_rows = []
split_sizes = {}
for name, (x, y, a) in datasets.items():
    indices = np.arange(len(y))
    train_idx, test_idx = train_test_split(
        indices, test_size=TEST_SIZE, random_state=SEED, stratify=2 * a + y,
    )
    assert not np.intersect1d(train_idx, test_idx).size
    split_sizes[name] = (len(train_idx), len(test_idx))
    y_train, y_test = y[train_idx].copy(), y[test_idx].copy()
    a_train, a_test = a[train_idx], a[test_idx]
    preprocessor = make_preprocessor(x)
    x_train = preprocessor.fit_transform(x.iloc[train_idx])
    x_test = preprocessor.transform(x.iloc[test_idx])
    test_neighbours = neighbours(x_test)

    weights = reweighing_weights(y_train, a_train)
    rng = np.random.default_rng(SEED)
    sampled = rng.choice(len(y_train), size=len(y_train), replace=True, p=weights / weights.sum())
    training_sets = {
        "Before mitigation": (x_train, y_train),
        "After Reweighing": (x_train[sampled], y_train[sampled]),
    }
    for model_name, (features, labels) in training_sets.items():
        model = LogisticRegression(C=1.0, solver="liblinear", max_iter=2000, random_state=SEED)
        model.fit(features, labels)
        positive_index = list(model.classes_).index(1)
        prediction = (model.predict_proba(x_test)[:, positive_index] >= 0.5).astype(int)
        test_rows.append({
            "Dataset": name,
            "Model": model_name,
            **fairness_scores(prediction, a_test, test_neighbours),
            "Accuracy": accuracy_score(y_test, prediction),
        })
    # The input labels are still the original labels.
    assert np.array_equal(y_train, y[train_idx])
    assert np.array_equal(y_test, y[test_idx])

test_results = pd.DataFrame(test_rows)
display(test_results.round(4))

,Dataset,Model,DP_Diff,DIR,Consistency,Accuracy
0,Adult,Before mitigation,-0.1726,0.3175,0.9645,0.8522
1,Adult,After Reweighing,-0.0826,0.6048,0.9636,0.8464
2,German Credit,Before mitigation,-0.0117,0.9845,0.8273,0.7567
3,German Credit,After Reweighing,-0.0090,0.9876,0.8120,0.7600
4,COMPAS,Before mitigation,-0.2559,0.6579,0.9802,0.6673
5,COMPAS,After Reweighing,-0.2602,0.6608,0.9778,0.6660


## 4 Results and discussion / 结果与讨论

The first table gives the required scores for the original labels. The second table gives scores for predictions on held-out records. We compare these stages below. The two test models also give a direct comparison on the same records. Original-label and test consistency use different neighbour sets, so the test-model comparison is the clearer check of the effect of Reweighing.

第一张表给出原始标签的指标，第二张表给出保留测试记录上的预测指标。下面比较这两个阶段，同时用同一测试集上的两个模型直接比较算法影响。原始标签和测试预测的一致性使用不同的邻居集合，因此两个测试模型之间的比较更适合评估重新加权的效果。

### Adult

From original labels to final predictions, DP_Diff changed from **-0.1945 to -0.0826**, DIR from **0.3597 to 0.6048**, and consistency from **0.8432 to 0.9636**. The test-model comparison also showed a smaller group gap. Absolute DP_Diff fell from **0.1726 to 0.0826**, and DIR rose from **0.3175 to 0.6048**. Accuracy fell from **85.22% to 84.64%**. Test consistency changed only slightly, from **0.9645 to 0.9636**. In this run, Reweighing improved group parity with a small accuracy loss. A group gap remained.

从原始标签到最终预测，DP_Diff 从 **-0.1945 变为 -0.0826**，DIR 从 **0.3597 变为 0.6048**，一致性从 **0.8432 变为 0.9636**。同一测试集的模型比较也显示群体差距缩小，DP_Diff 的绝对值从 **0.1726 降至 0.0826**，DIR 从 **0.3175 升至 0.6048**。准确率从 **85.22% 降至 84.64%**，测试预测一致性从 **0.9645 略降至 0.9636**。这次运行以少量准确率下降换来了群体公平性的改善，但仍有群体差距。

### German Credit

From original labels to final predictions, DP_Diff changed from **-0.0748 to -0.0090**, DIR from **0.8966 to 0.9876**, and consistency from **0.7428 to 0.8120**. The ordinary test model already had a small gap. Reweighing changed absolute DP_Diff from **0.0117 to 0.0090** and DIR from **0.9845 to 0.9876**. Accuracy changed from **75.67% to 76.00%**, while test consistency fell from **0.8273 to 0.8120**. The changes in group scores and accuracy were small. With only 300 test records, this run gives limited evidence for a clear benefit.

从原始标签到最终预测，DP_Diff 从 **-0.0748 变为 -0.0090**，DIR 从 **0.8966 变为 0.9876**，一致性从 **0.7428 变为 0.8120**。普通测试模型的群体差距本来就很小。重新加权让 DP_Diff 的绝对值从 **0.0117 变为 0.0090**，DIR 从 **0.9845 变为 0.9876**。准确率从 **75.67% 变为 76.00%**，测试一致性从 **0.8273 降至 0.8120**。群体指标与准确率变化都很小，而且测试集只有 300 条记录，本次结果对明确改善的支持有限。

### COMPAS

From original labels to final predictions, DP_Diff changed from **-0.1323 to -0.2602** and DIR from **0.7828 to 0.6608**, so the group gap grew. Consistency rose from **0.6529 to 0.9778**. Against the ordinary test model, Reweighing slightly increased absolute DP_Diff from **0.2559 to 0.2602**. DIR moved slightly closer to 1, from **0.6579 to 0.6608**. These two group scores changed differently because DIR also depends on the reference group's positive rate. Accuracy fell from **66.73% to 66.60%**, and test consistency from **0.9802 to 0.9778**. Reweighing gave no clear overall improvement here.

从原始标签到最终预测，DP_Diff 从 **-0.1323 变为 -0.2602**，DIR 从 **0.7828 变为 0.6608**，群体差距扩大。一致性则从 **0.6529 升至 0.9778**。与普通测试模型相比，重新加权使 DP_Diff 的绝对值从 **0.2559 略增至 0.2602**，DIR 从 **0.6579 略升至 0.6608**，稍微接近 1。两项群体指标的变化方向不同，是因为 DIR 还取决于参照组的正类比例。准确率从 **66.73% 降至 66.60%**，测试一致性从 **0.9802 降至 0.9778**。本次实验没有显示出明确的总体改善。

### Interpretation and limits / 解释与局限

The exact training weights equalise the weighted positive rates before sampling. Future prediction rates can still differ, as COMPAS shows. Reweighing targets group parity, so it may not improve neighbour consistency. High consistency can also occur when a model makes the same prediction for many records. Results depend on the chosen features, neighbour distance, classifier, random sample, and split. We used one fixed split and one fixed resampling seed, so these results do not establish that one method will always perform better. The scores describe the recorded labels and this model, and do not establish fairness in every sense.

精确训练权重使抽样前的加权群体正类比例相等，但未来预测比例仍可能不同，COMPAS 就体现了这一点。Reweighing 针对群体比例公平，未必改善邻居一致性。当模型给许多记录相同预测时，一致性也可能较高。结果受特征、邻居距离、分类器、随机抽样和数据划分影响。本实验只用一个固定划分和一个抽样种子，因此无法证明某个方法总是更好。这些分数描述记录中的标签和当前模型，不能覆盖所有公平性含义。

## Sources / 来源

1. Calders, Kamiran and Pechenizkiy (2009), *Building Classifiers with Independency Constraints*, Section 3.2. [Paper DOI](https://doi.org/10.1109/ICDMW.2009.83).
2. [UCI Adult](https://archive.ics.uci.edu/dataset/2/adult). [UCI German Credit](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data). Original data and codebook are included in the downloaded archives.
3. [ProPublica COMPAS data](https://github.com/propublica/compas-analysis) and [analysis notebook with the filtering rules](https://github.com/propublica/compas-analysis/blob/master/Compas%20Analysis.ipynb).
4. [AIF360 metric definitions](https://aif360.readthedocs.io/en/latest/modules/generated/aif360.metrics.BinaryLabelDatasetMetric.html) and [consistency implementation](https://github.com/Trusted-AI/AIF360/blob/main/aif360/metrics/binary_label_dataset_metric.py). We implement the formulas directly with NumPy and scikit-learn.